# BulkRNABert: Three Representation Levels

This notebook extracts and documents three representations from the pretrained **BulkRNABert-TCGA** checkpoint:

1. **Static gene embeddings**: one learned 200-dimensional vector per gene.
2. **Contextual gene embeddings**: one 256-dimensional vector per gene, conditioned on a particular bulk RNA-seq sample.
3. **Sample embeddings**: one 256-dimensional vector per sample, obtained by mean-pooling the final contextual gene embeddings across genes.

The notebook uses the repository's bundled `tcga_sample.csv`, pretrained checkpoint, tokenizer, and fixed gene ordering.


## Expected repository layout

Run this notebook from the root of `BulkRNABert_multiomics-open-research_fork`.

Required files:

- `data/bulkrnabert/tcga_sample.csv`
- `data/bulkrnabert/common_gene_id.txt`
- `checkpoints/bulk_rna_bert_tcga/config.json`
- `checkpoints/bulk_rna_bert_tcga/params.joblib`

The repository environment used during validation contained JAX 0.4.19, Haiku 0.0.10, NumPy 1.26.0, and pandas 2.2.0.


In [1]:
from pathlib import Path
import time

import haiku as hk
import jax
import jax.numpy as jnp
import joblib
import numpy as np
import pandas as pd

from multiomics_open_research.bulk_rna_bert.pretrained import (
    get_bulkrnabert_pretrained_model,
)

ROOT = Path.cwd()
DATA_FILE = ROOT / "data" / "bulkrnabert" / "tcga_sample.csv"
GENE_FILE = ROOT / "data" / "bulkrnabert" / "common_gene_id.txt"
CHECKPOINT_DIR = ROOT / "checkpoints" / "bulk_rna_bert_tcga"

OUTPUT_DIR = ROOT / "BulkRNABert_representations"
SNAPSHOT_DIR = OUTPUT_DIR / "snapshots"
FULL_DIR = OUTPUT_DIR / "full_outputs"

SNAPSHOT_DIR.mkdir(parents=True, exist_ok=True)
FULL_DIR.mkdir(parents=True, exist_ok=True)

print("Repository root:", ROOT)
print("Output directory:", OUTPUT_DIR)


Repository root: e:\BulkRNABert_multiomics-open-research_fork
Output directory: e:\BulkRNABert_multiomics-open-research_fork\BulkRNABert_representations


C:\Users\Genesilico-FM\AppData\Local\Temp\ipykernel_78500\3509213524.py:9: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


## 1. Load the pretrained BulkRNABert-TCGA model


In [2]:
params, forward_fn, tokenizer, config = get_bulkrnabert_pretrained_model(
    model_name="bulk_rna_bert_tcga",
    embeddings_layers_to_save=(4,),
)

print("Genes:", config.n_genes)
print("Expression bins:", config.n_expressions_bins)
print("Transformer embedding dimension:", config.embed_dim)
print("Initial gene embedding dimension:", config.init_gene_embed_dim)
print("Transformer layers:", config.num_layers)
print("Attention heads:", config.num_attention_heads)
print("Project gene embedding:", config.project_gene_embedding)
print("Log normalization:", config.use_log_normalization)
print("Max normalization:", config.use_max_normalization)
print("Normalization factor:", config.normalization_factor)


Genes: 19062
Expression bins: 64
Transformer embedding dimension: 256
Initial gene embedding dimension: 200
Transformer layers: 4
Attention heads: 8
Project gene embedding: True
Log normalization: True
Max normalization: True
Normalization factor: 5.547176906585117


The validated TCGA checkpoint uses 19,062 genes, a 200-dimensional static gene embedding, and a 256-dimensional transformer representation. The model has four transformer layers.


## 2. Load the bundled TCGA expression and survival example


In [3]:
df = pd.read_csv(DATA_FILE)

metadata_cols = ["identifier", "survival_time", "event"]
expression_df = df.drop(columns=metadata_cols)

print("Dataset shape:", df.shape)
print("Expression shape:", expression_df.shape)
print()
print(df[metadata_cols].to_string(index=False))

assert expression_df.shape[1] == config.n_genes


Dataset shape: (4, 19065)
Expression shape: (4, 19062)

      identifier  survival_time  event
TCGA-06-2559-01A            150      1
TCGA-06-0187-01A            828      1
TCGA-27-2521-01A            510      1
TCGA-06-0744-01A           1426      1


Each row is one TCGA tumor sample. The first three columns contain the sample identifier and survival outcome. The remaining 19,062 columns are gene-expression values in the fixed BulkRNABert gene order.


## 3. Load and verify the fixed gene ordering


In [4]:
with open(GENE_FILE, "r", encoding="utf-8") as f:
    gene_ids = [line.strip() for line in f if line.strip()]

print("Gene IDs loaded:", len(gene_ids))
print("First 10 gene IDs:", gene_ids[:10])
print("First 10 expression columns:", expression_df.columns[:10].tolist())

assert len(gene_ids) == config.n_genes

if gene_ids == expression_df.columns.tolist():
    print("Gene ordering exactly matches tcga_sample.csv.")
else:
    print("Gene list and expression-column ordering are not text-identical.")
    print("The repository data should still be kept in its supplied fixed order.")


Gene IDs loaded: 19062
First 10 gene IDs: ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419', 'ENSG00000000457', 'ENSG00000000460', 'ENSG00000000938', 'ENSG00000000971', 'ENSG00000001036', 'ENSG00000001084', 'ENSG00000001167']
First 10 expression columns: ['ENSG00000000003', 'ENSG00000000005', 'ENSG00000000419', 'ENSG00000000457', 'ENSG00000000460', 'ENSG00000000938', 'ENSG00000000971', 'ENSG00000001036', 'ENSG00000001084', 'ENSG00000001167']
Gene ordering exactly matches tcga_sample.csv.


## 4. Preprocess and tokenize the TCGA expression matrix


In [5]:
expression = expression_df.to_numpy(dtype=np.float32)
x = jnp.asarray(expression)

# Repository inference preprocessing: log10(1 + expression)
x_log = jnp.log10(1.0 + x)

tokens = tokenizer.batch_tokenize(x_log)

print("Raw expression:", expression.shape)
print("Log-transformed expression:", x_log.shape)
print("Tokens:", tokens.shape)
print("Token range:", int(tokens.min()), "to", int(tokens.max()))
print("First sample, first 20 tokens:")
print(tokens[0, :20])


Raw expression: (4, 19062)
Log-transformed expression: (4, 19062)
Tokens: (4, 19062)
Token range: 0 to 64
First sample, first 20 tokens:
[21  2 24 12 12 15 11 21 15 18  3 10 15 11 13  6 22  8 11 16]


## 5. Static gene embeddings

The checkpoint contains a learned `gene_embedding` parameter with shape `19,062 × 200`. This representation is sample-independent: each gene has one static vector.

Because the checkpoint uses `project_gene_embedding=True`, the 200-dimensional vectors are projected to 256 dimensions before being added to expression-token embeddings. We retain the raw 200-dimensional matrix as the static gene representation and also compute the projected model-input form.


In [6]:
gene_embedding_key = "bulk_bert/~/gene_embedding"
projection_key = "bulk_bert/~/linear"

static_gene_embeddings = np.asarray(
    params[gene_embedding_key]["embeddings"]
)

projection_w = np.asarray(params[projection_key]["w"])
projection_b = np.asarray(params[projection_key]["b"])

projected_gene_embeddings = (
    static_gene_embeddings @ projection_w + projection_b
)

print("Static gene embeddings:", static_gene_embeddings.shape)
print("Projection weights:", projection_w.shape)
print("Projected gene embeddings:", projected_gene_embeddings.shape)

assert static_gene_embeddings.shape == (
    config.n_genes,
    config.init_gene_embed_dim,
)
assert projected_gene_embeddings.shape == (
    config.n_genes,
    config.embed_dim,
)


Static gene embeddings: (19062, 200)
Projection weights: (200, 256)
Projected gene embeddings: (19062, 256)


In [7]:
# Full machine-readable outputs
np.save(
    FULL_DIR / "static_gene_embeddings.npy",
    static_gene_embeddings,
)
np.save(
    FULL_DIR / "projected_static_gene_embeddings.npy",
    projected_gene_embeddings,
)

# Readable snapshot: first 10 genes × first 10 dimensions
static_snapshot = pd.DataFrame(
    static_gene_embeddings[:10, :10],
    columns=[f"dim_{i+1}" for i in range(10)],
)
static_snapshot.insert(0, "Gene_ID", gene_ids[:10])

static_snapshot.to_csv(
    SNAPSHOT_DIR / "01_static_gene_embedding_snapshot.csv",
    index=False,
)

static_snapshot


,Gene_ID,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10
0,ENSG00000000003,0.047667,0.175386,-0.386181,-0.123492,0.259778,0.409755,0.075362,-0.007544,0.254885,-0.307572
1,ENSG00000000005,-0.398308,-0.352640,0.272732,0.428776,-0.289421,0.284004,-0.042088,0.087365,0.148132,-0.190575
2,ENSG00000000419,0.110905,0.237906,-0.021752,-0.018782,-0.293922,0.024809,-0.140693,0.195797,0.220049,0.039049
3,ENSG00000000457,-0.394273,0.027533,0.210927,-0.112010,0.219379,0.173738,0.155192,-0.323743,0.264836,0.000300
4,ENSG00000000460,-0.011197,0.287259,0.115467,-0.038219,0.256914,0.594860,0.111140,-0.333544,0.741817,0.239168
5,ENSG00000000938,0.153805,0.454422,-0.095107,-0.020031,0.132693,0.039416,0.325032,-0.164929,0.147032,-0.391874
6,ENSG00000000971,0.241920,0.121870,0.235494,0.179737,-0.041686,0.194905,0.439652,0.668278,0.108032,-0.189008
7,ENSG00000001036,0.235991,0.445458,-0.375298,0.201003,0.224115,-0.162673,-0.253785,0.430675,0.232374,-0.070450
8,ENSG00000001084,0.154632,-0.000954,0.130885,-0.505255,0.239240,-0.005476,0.157619,0.104622,-0.115985,-0.255767
9,ENSG00000001167,0.293279,0.148825,0.068516,-0.103912,-0.051373,-0.192211,0.082150,-0.228514,0.281847,-0.273582


## 6. Run BulkRNABert and extract contextual gene embeddings

We request `embeddings_4`, the output of the fourth and final transformer layer. Its tensor layout is:

`sample × gene × embedding dimension`

For these four bundled TCGA samples, the expected shape is `4 × 19,062 × 256`.


In [8]:
transformed = hk.transform(forward_fn)
apply_fn = jax.jit(transformed.apply)

outputs = apply_fn(
    params,
    None,
    tokens,
)

print("Output keys:", list(outputs.keys()))

for key, value in outputs.items():
    if hasattr(value, "shape"):
        print(f"{key}: {value.shape}")

contextual_gene_embeddings = np.asarray(outputs["embeddings_4"])

assert contextual_gene_embeddings.shape == (
    len(df),
    config.n_genes,
    config.embed_dim,
)

print("Contextual gene embeddings:", contextual_gene_embeddings.shape)


Output keys: ['embeddings_4', 'logits']
embeddings_4: (4, 19062, 256)
logits: (4, 19062, 64)
Contextual gene embeddings: (4, 19062, 256)


In [9]:
np.save(
    FULL_DIR / "contextual_gene_embeddings.npy",
    contextual_gene_embeddings,
)

# Snapshot: first 10 genes for every bundled TCGA sample,
# retaining the first 10 embedding dimensions.
context_rows = []

for sample_idx, sample_id in enumerate(df["identifier"]):
    for gene_idx in range(10):
        row = {
            "Sample_ID": sample_id,
            "Gene_ID": gene_ids[gene_idx],
        }
        for dim in range(10):
            row[f"dim_{dim+1}"] = contextual_gene_embeddings[
                sample_idx, gene_idx, dim
            ]
        context_rows.append(row)

contextual_snapshot = pd.DataFrame(context_rows)

contextual_snapshot.to_csv(
    SNAPSHOT_DIR / "02_contextual_gene_embedding_snapshot.csv",
    index=False,
)

contextual_snapshot.head(20)


,Sample_ID,Gene_ID,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10
0,TCGA-06-2559-01A,ENSG00000000003,-3.278287,0.207391,-3.373292,0.228223,-0.558985,-0.565454,-0.859745,0.897572,-0.549189,-1.096231
1,TCGA-06-2559-01A,ENSG00000000005,-0.230596,0.579220,3.780765,-0.110427,-0.428184,1.681638,-2.373319,-0.016836,1.254593,-0.719007
2,TCGA-06-2559-01A,ENSG00000000419,-0.089846,-2.904783,-0.380949,-0.620145,0.430434,-1.607608,-2.868238,-0.159184,1.271490,-1.841574
3,TCGA-06-2559-01A,ENSG00000000457,-1.478216,2.973970,2.486674,-1.664442,0.404989,2.972106,3.743552,0.211318,2.986017,1.593617
4,TCGA-06-2559-01A,ENSG00000000460,0.444747,3.384552,0.379649,-0.715482,-0.224215,3.062737,3.874532,-1.006958,1.504379,1.652934
5,TCGA-06-2559-01A,ENSG00000000938,-0.413261,0.028857,-5.143152,2.769716,-1.228695,0.778828,0.192257,-2.414982,1.922750,1.602402
6,TCGA-06-2559-01A,ENSG00000000971,3.181018,-1.986271,1.475428,-0.885952,-0.224171,-2.153929,-0.224768,0.910751,-0.135564,1.357133
7,TCGA-06-2559-01A,ENSG00000001036,-0.438290,1.886745,-5.663216,1.399715,-0.970062,0.418266,0.389574,0.507687,-0.827102,-0.349305
8,TCGA-06-2559-01A,ENSG00000001084,-1.076252,-0.730673,-4.409523,-0.101024,1.201878,2.771971,-0.095686,-1.986867,3.158288,-0.177222
9,TCGA-06-2559-01A,ENSG00000001167,-0.230233,-2.639651,-2.407632,-0.027034,-0.260039,1.394842,-0.884335,-1.123944,2.871405,2.932127


### Demonstrate contextuality

The same gene has one static embedding, while its final transformer embedding can differ between samples. The table below shows the first gene across all four TCGA samples.


In [10]:
first_gene_comparison = contextual_snapshot[
    contextual_snapshot["Gene_ID"] == gene_ids[0]
].copy()

first_gene_comparison


,Sample_ID,Gene_ID,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,dim_8,dim_9,dim_10
0,TCGA-06-2559-01A,ENSG00000000003,-3.278287,0.207391,-3.373292,0.228223,-0.558985,-0.565454,-0.859745,0.897572,-0.549189,-1.096231
10,TCGA-06-0187-01A,ENSG00000000003,-1.270687,-1.482020,1.434989,-0.825429,2.993618,-0.467648,-1.485002,-0.260763,0.617894,0.967208
20,TCGA-27-2521-01A,ENSG00000000003,-2.385746,-0.620836,-0.730000,-0.977844,-0.799002,0.037583,-2.102263,1.327631,-0.238386,-0.098714
30,TCGA-06-0744-01A,ENSG00000000003,-3.536766,0.162165,-3.064801,0.267244,0.010706,-0.442786,-1.174531,1.334857,-0.198520,-0.883097


## 7. Sample embeddings

The repository downstream head receives embeddings with shape `(batch, seq_length, emb_dim)`, transposes them, and computes a mean across the sequence dimension. Equivalently, we can mean-pool `embeddings_4` across the gene axis.

The resulting representation has one 256-dimensional vector per bulk RNA-seq sample.


In [11]:
sample_embeddings = contextual_gene_embeddings.mean(axis=1)

print("Sample embeddings:", sample_embeddings.shape)

assert sample_embeddings.shape == (
    len(df),
    config.embed_dim,
)

np.save(
    FULL_DIR / "sample_embeddings.npy",
    sample_embeddings,
)


Sample embeddings: (4, 256)


In [12]:
sample_snapshot = pd.DataFrame(
    sample_embeddings[:, :20],
    columns=[f"dim_{i+1}" for i in range(20)],
)

sample_snapshot.insert(0, "event", df["event"].to_numpy())
sample_snapshot.insert(
    0,
    "survival_time",
    df["survival_time"].to_numpy(),
)
sample_snapshot.insert(
    0,
    "Sample_ID",
    df["identifier"].to_numpy(),
)

sample_snapshot.to_csv(
    SNAPSHOT_DIR / "03_sample_embedding_snapshot.csv",
    index=False,
)

sample_snapshot


,Sample_ID,survival_time,event,dim_1,dim_2,dim_3,dim_4,dim_5,dim_6,dim_7,...,dim_11,dim_12,dim_13,dim_14,dim_15,dim_16,dim_17,dim_18,dim_19,dim_20
0,TCGA-06-2559-01A,150,1,0.531084,0.075954,0.780909,0.168625,0.362793,0.051320,-0.625300,...,-0.137407,0.290510,0.321484,-1.666211,0.678940,0.138059,0.331935,-0.286606,0.115098,-0.056302
1,TCGA-06-0187-01A,828,1,0.367409,-0.101660,1.489338,0.300650,0.588334,-0.032680,-0.626450,...,-0.081314,0.111633,0.213540,-1.975140,0.846386,0.294525,0.319014,-0.328362,0.197260,0.017000
2,TCGA-27-2521-01A,510,1,0.381403,-0.130095,1.348344,0.243466,0.365977,-0.086192,-0.624271,...,-0.208731,0.174666,0.253649,-2.166295,0.754334,0.038231,0.282698,-0.348401,0.045971,0.122107
3,TCGA-06-0744-01A,1426,1,0.392257,-0.023996,1.323607,0.343914,0.368766,-0.163262,-0.589156,...,-0.062803,0.157803,0.234954,-1.861360,0.913689,0.247678,0.329065,-0.378919,0.166606,0.163369


## 8. Raw expression + survival snapshot


In [13]:
# Keep a small readable source-data snapshot as well.
# First 10 genes for all four bundled TCGA samples.

raw_snapshot = df[
    metadata_cols + expression_df.columns[:10].tolist()
].copy()

raw_snapshot.to_csv(
    SNAPSHOT_DIR / "00_tcga_expression_survival_snapshot.csv",
    index=False,
)

raw_snapshot


,identifier,survival_time,event,ENSG00000000003,ENSG00000000005,ENSG00000000419,ENSG00000000457,ENSG00000000460,ENSG00000000938,ENSG00000000971,ENSG00000001036,ENSG00000001084,ENSG00000001167
0,TCGA-06-2559-01A,150,1,69.5439,0.4599,112.2372,8.4532,8.4303,19.1823,7.8015,62.8843,19.1064,31.7049
1,TCGA-06-0187-01A,828,1,55.2986,1.8746,51.8722,4.8353,4.0387,12.6223,12.9179,38.7700,12.1175,15.9724
2,TCGA-27-2521-01A,510,1,5.8836,1.3023,115.9459,6.9493,4.3589,6.8639,2.4921,30.7555,17.7058,19.6703
3,TCGA-06-0744-01A,1426,1,60.4576,0.3522,72.5223,4.7773,2.6995,5.7645,7.3780,49.9015,10.4739,12.1921


## 9. Representation summary


In [14]:
summary = pd.DataFrame(
    [
        {
            "Representation": "Static gene embedding",
            "Source": "Checkpoint gene_embedding parameter",
            "Shape": str(static_gene_embeddings.shape),
            "Sample-dependent": "No",
            "Dimension": config.init_gene_embed_dim,
        },
        {
            "Representation": "Projected static gene embedding",
            "Source": "Static gene embedding + learned linear projection",
            "Shape": str(projected_gene_embeddings.shape),
            "Sample-dependent": "No",
            "Dimension": config.embed_dim,
        },
        {
            "Representation": "Contextual gene embedding",
            "Source": "Final transformer layer: embeddings_4",
            "Shape": str(contextual_gene_embeddings.shape),
            "Sample-dependent": "Yes",
            "Dimension": config.embed_dim,
        },
        {
            "Representation": "Sample embedding",
            "Source": "Mean of final contextual embeddings across genes",
            "Shape": str(sample_embeddings.shape),
            "Sample-dependent": "Yes",
            "Dimension": config.embed_dim,
        },
    ]
)

summary


,Representation,Source,Shape,Sample-dependent,Dimension
0,Static gene embedding,Checkpoint gene_embedding parameter,"(19062, 200)",No,200
1,Projected static gene embedding,Static gene embedding + learned linear projection,"(19062, 256)",No,256
2,Contextual gene embedding,Final transformer layer: embeddings_4,"(4, 19062, 256)",Yes,256
3,Sample embedding,Mean of final contextual embeddings across genes,"(4, 256)",Yes,256


## 10. Verify saved files


In [15]:
print("Snapshots:")
for path in sorted(SNAPSHOT_DIR.glob("*")):
    print(" ", path.relative_to(ROOT))

print("\nFull outputs:")
for path in sorted(FULL_DIR.glob("*")):
    size_mb = path.stat().st_size / (1024 ** 2)
    print(f"  {path.relative_to(ROOT)}  ({size_mb:.2f} MB)")


Snapshots:
  BulkRNABert_representations\snapshots\00_tcga_expression_survival_snapshot.csv
  BulkRNABert_representations\snapshots\01_static_gene_embedding_snapshot.csv
  BulkRNABert_representations\snapshots\02_contextual_gene_embedding_snapshot.csv
  BulkRNABert_representations\snapshots\03_sample_embedding_snapshot.csv

Full outputs:
  BulkRNABert_representations\full_outputs\contextual_gene_embeddings.npy  (74.46 MB)
  BulkRNABert_representations\full_outputs\projected_static_gene_embeddings.npy  (18.62 MB)
  BulkRNABert_representations\full_outputs\sample_embeddings.npy  (0.00 MB)
  BulkRNABert_representations\full_outputs\static_gene_embeddings.npy  (14.54 MB)


## Interpretation

**Static gene embedding:** a single 200-dimensional learned vector for each of the 19,062 fixed genes. It does not change across patients.

**Contextual gene embedding:** a 256-dimensional final-transformer vector for each gene in each sample. The representation can change between patients because it is conditioned on the complete tokenized transcriptomic context.

**Sample embedding:** a 256-dimensional vector obtained by mean-pooling the final contextual gene embeddings across the 19,062 gene positions. This is the representation consumed by the repository's downstream MLP-style head.

The bundled TCGA example also provides `survival_time` and `event`, allowing the source expression data and sample representation to be connected to a downstream survival target.


## 11. Runtime and GPU-memory benchmark

This section records the requested comparison columns:

`Tool | Embedding type | Dimension | Output | Method | Time | Peak GPU VRAM`

Timing uses JAX synchronization with `block_until_ready()` so asynchronous device execution is included. GPU memory is queried through `nvidia-smi` when JAX is running on a GPU and NVIDIA tooling is available. On CPU, GPU VRAM is reported as `N/A (CPU)`.

The contextual benchmark runs BulkRNABert on one TCGA sample. The sample-embedding benchmark times mean pooling of that already-computed contextual representation. The static benchmark times retrieval of the learned checkpoint matrix.


In [16]:
import shutil
import subprocess

devices = jax.devices()
print("JAX devices:", devices)

running_on_gpu = any(d.platform == "gpu" for d in devices)
print("Running on GPU:", running_on_gpu)


def sync_array(x):
    # JAX executes asynchronously on accelerators.
    if hasattr(x, "block_until_ready"):
        x.block_until_ready()
    return x


def current_gpu_memory_mib():
    if not running_on_gpu:
        return None

    if shutil.which("nvidia-smi") is None:
        return None

    try:
        result = subprocess.run(
            [
                "nvidia-smi",
                "--query-compute-apps=used_memory",
                "--format=csv,noheader,nounits",
            ],
            capture_output=True,
            text=True,
            check=True,
        )

        values = []
        for line in result.stdout.splitlines():
            line = line.strip()
            if line:
                try:
                    values.append(float(line))
                except ValueError:
                    pass

        return max(values) if values else None

    except Exception:
        return None


JAX devices: [CpuDevice(id=0)]
Running on GPU: False


In [17]:
# ------------------------------------------------------------
# Static gene embedding benchmark
# ------------------------------------------------------------

mem_before_static = current_gpu_memory_mib()

t0 = time.perf_counter()

static_bench = np.asarray(
    params["bulk_bert/~/gene_embedding"]["embeddings"]
).copy()

static_time = time.perf_counter() - t0

mem_after_static = current_gpu_memory_mib()

if mem_before_static is not None and mem_after_static is not None:
    static_vram = max(mem_before_static, mem_after_static)
else:
    static_vram = None

print("Static embedding shape:", static_bench.shape)
print(f"Static retrieval time: {static_time:.6f} s")


Static embedding shape: (19062, 200)
Static retrieval time: 0.002956 s


In [18]:
# ------------------------------------------------------------
# Contextual gene embedding benchmark
# One TCGA sample through the full BulkRNABert transformer.
# ------------------------------------------------------------

one_sample_tokens = tokens[:1]

# Warm-up once so JIT compilation is not mixed with steady-state inference time.
warmup = apply_fn(params, None, one_sample_tokens)
sync_array(warmup["embeddings_4"])

mem_before_context = current_gpu_memory_mib()

t0 = time.perf_counter()

context_bench_outputs = apply_fn(
    params,
    None,
    one_sample_tokens,
)

context_bench = context_bench_outputs["embeddings_4"]
sync_array(context_bench)

context_time = time.perf_counter() - t0

mem_after_context = current_gpu_memory_mib()

if mem_before_context is not None and mem_after_context is not None:
    contextual_vram = max(mem_before_context, mem_after_context)
else:
    contextual_vram = None

print("Contextual embedding shape:", context_bench.shape)
print(f"Contextual inference time: {context_time:.6f} s")


Contextual embedding shape: (1, 19062, 256)
Contextual inference time: 23.144672 s


In [19]:
# ------------------------------------------------------------
# Sample embedding benchmark
# Mean pooling over the 19,062 contextual gene positions.
# ------------------------------------------------------------

context_for_pooling = context_bench

mem_before_sample = current_gpu_memory_mib()

t0 = time.perf_counter()

sample_bench = jnp.mean(
    context_for_pooling,
    axis=1,
)

sync_array(sample_bench)

sample_time = time.perf_counter() - t0

mem_after_sample = current_gpu_memory_mib()

if mem_before_sample is not None and mem_after_sample is not None:
    sample_vram = max(mem_before_sample, mem_after_sample)
else:
    sample_vram = None

print("Sample embedding shape:", sample_bench.shape)
print(f"Sample pooling time: {sample_time:.6f} s")


Sample embedding shape: (1, 256)
Sample pooling time: 0.029078 s


In [20]:
def format_vram(value):
    if not running_on_gpu:
        return "N/A (CPU)"
    if value is None:
        return "N/A"
    return f"{value:.1f} MiB"


benchmark_summary = pd.DataFrame(
    [
        {
            "Tool": "BulkRNABert",
            "Embedding type": "Static gene",
            "Dimension": 200,
            "Output": "19,062 gene embeddings",
            "Method": "Checkpoint gene_embedding lookup",
            "Time": f"{static_time:.6f} s",
            "Peak GPU VRAM": format_vram(static_vram),
        },
        {
            "Tool": "BulkRNABert",
            "Embedding type": "Contextual gene",
            "Dimension": 256,
            "Output": "1 × 19,062 contextual gene embeddings",
            "Method": "Final transformer layer (embeddings_4)",
            "Time": f"{context_time:.6f} s",
            "Peak GPU VRAM": format_vram(contextual_vram),
        },
        {
            "Tool": "BulkRNABert",
            "Embedding type": "Sample",
            "Dimension": 256,
            "Output": "1 sample embedding",
            "Method": "Mean pooling across contextual gene embeddings",
            "Time": f"{sample_time:.6f} s",
            "Peak GPU VRAM": format_vram(sample_vram),
        },
    ]
)

benchmark_summary.to_csv(
    OUTPUT_DIR / "benchmark_summary.csv",
    index=False,
)

benchmark_summary


,Tool,Embedding type,Dimension,Output,Method,Time,Peak GPU VRAM
0,BulkRNABert,Static gene,200,"19,062 gene embeddings",Checkpoint gene_embedding lookup,0.002956 s,N/A (CPU)
1,BulkRNABert,Contextual gene,256,"1 × 19,062 contextual gene embeddings",Final transformer layer (embeddings_4),23.144672 s,N/A (CPU)
2,BulkRNABert,Sample,256,1 sample embedding,Mean pooling across contextual gene embeddings,0.029078 s,N/A (CPU)


### Benchmark interpretation

The static row measures retrieval of the learned static gene matrix. The contextual row measures steady-state inference for one sample after a JIT warm-up. The sample row measures only the pooling step applied to an existing contextual representation.

`nvidia-smi` reports process-level memory usage rather than an exact per-operation allocation. Treat the VRAM column as an observed process-memory benchmark for the run. For rigorous GPU peak-memory profiling, use an accelerator-specific profiler and a controlled fresh process per representation.


## 12. Final output structure

After running the notebook, the generated representation directory contains:

- `snapshots/00_tcga_expression_survival_snapshot.csv`
- `snapshots/01_static_gene_embedding_snapshot.csv`
- `snapshots/02_contextual_gene_embedding_snapshot.csv`
- `snapshots/03_sample_embedding_snapshot.csv`
- `full_outputs/static_gene_embeddings.npy`
- `full_outputs/projected_static_gene_embeddings.npy`
- `full_outputs/contextual_gene_embeddings.npy`
- `full_outputs/sample_embeddings.npy`
- `benchmark_summary.csv`
